# BERT Text Classifier

This notebook fine-tunes a **BERT** (`bert-base-uncased`) sequence-classification model on an email dataset to separate phishing emails from safe ones. It then evaluates the model and derives a **three-class** decision rule (safe / human review / phishing) from two calibrated thresholds.

## Pipeline

1. Load the phishing email dataset from Hugging Face.
2. Tokenize with the BERT tokenizer and wrap it in a `torch` `Dataset`.
3. Fine-tune `BertForSequenceClassification` for 3 epochs.
4. Save the model to `./bert_test`.
5. Evaluate (accuracy, precision, recall, F1, AUC and a confusion matrix).
6. Derive lower/upper thresholds from the precision-recall curve.
7. Classify example emails with the three-class rule.


## 1. Setup and imports


In [4]:
import pandas as pd
!pip install transformers datasets scikit-learn torch --quiet
!pip install -U transformers --quiet
!pip install 'accelerate>=0.26.0'
import transformers
print(transformers.__version__)
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import precision_recall_curve, roc_auc_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
from transformers import BertForSequenceClassification, BertTokenizer
import torch
from transformers import BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score

from torch.utils.data import Dataset


4.56.2


## 2. Load and explore the dataset


In [5]:
df = pd.read_csv("hf://datasets/zefang-liu/phishing-email-dataset/Phishing_Email.csv") # read the dataset
df.head(5) # display the first 5 rows of the dataset for testing purposes
df.columns = df.columns.str.strip() # remove any leading or trailing whitespace from the column names
print(df.columns.tolist()) # print the column names 

['Unnamed: 0', 'Email Text', 'Email Type']


In [6]:
print(df["Email Type"].value_counts())


Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64


In [7]:
# Normalize Email Type column
df["Email Type"] = df["Email Type"].str.strip()

# Create binary labels for classification
label_map = {
    "Safe Email": 0, 
    "Phishing Email": 1
}

df["label"] = df["Email Type"].map(label_map)

# Clean up the dataset
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)
df = df.drop(columns=["Unnamed: 0", "Email Type"])

# Split dataset into training and testing sets

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)


## 3. Tokenization and dataset


In [8]:


# Load pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Define custom dataset class for email classification
class PhishingDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        """Initialize the dataset with email texts and labels"""
        self.texts = df["Email Text"].astype(str).tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        """Return the total number of emails in the dataset"""
        return len(self.texts)

    def __getitem__(self, idx):
        """Get a single email and its label, tokenized for BERT"""
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Create training and testing datasets
train_dataset = PhishingDataset(train_df, tokenizer)
test_dataset = PhishingDataset(test_df, tokenizer)

## 4. Fine-tune BERT


In [ ]:

# Load pre-trained BERT model for binary classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Configure training parameters
training_args = TrainingArguments(
    output_dir="./bert_phishing",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
)

# Set up data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred): 
    """Calculate accuracy from model predictions"""
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

# Initialize the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Start training the model
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NameError: name 'tokenizer' is not defined

## 5. Save the fine-tuned model


In [9]:
output_dir = "./bert_test"

# Save the trained model and tokenizer
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")

NameError: name 'trainer' is not defined

## 6. Quick sanity check


In [9]:
model_path = "./bert_test"

# Load the trained model and tokenizer
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

# Test email for classification
test_email = "Hi, Thank you for showing interest in hiring me. I completed my Bachelor's in Software Engineering and I'm completing my Master's now in Cyber Security."

# Tokenize the email text
inputs = tokenizer(
    test_email,
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt"
)

# Make prediction
with torch.no_grad():
    outputs = model(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=1)
    phishing_probability = probabilities[0][1].item()

print(f"Phishing probability: {phishing_probability:.4f}")
if phishing_probability > 0.5: 
    print("Result: PHISHING EMAIL")
else:
    print("Result: SAFE EMAIL")

Phishing probability: 0.0049
Result: SAFE EMAIL


## 7. Evaluate the model


In [11]:

# define a function to evaluate the model
def evaluate_bert_model(model, tokenizer, test_dataset, test_df): 
 
    print("Model's evaluation")
      
    model.eval() # pytorch function to set the model to evaluation mode
    all_predictions = [] 
    all_probabilities = [] 
    all_labels = [] 
        
    # loop through the test dataset
    for i in range(len(test_dataset)): 
        item = test_dataset[i]
        inputs = {
            'input_ids': item['input_ids'].unsqueeze(0), 
            'attention_mask': item['attention_mask'].unsqueeze(0), 
            'token_type_ids': item['token_type_ids'].unsqueeze(0) if 'token_type_ids' in item else None 
        }
        
    
        inputs = {k: v for k, v in inputs.items() if v is not None} 
        
        with torch.no_grad(): # disable gradient calculation
            outputs = model(**inputs) 
            probabilities = torch.softmax(outputs.logits, dim=1) 
            phishing_prob = probabilities[0][1].item() 
            prediction = 1 if phishing_prob > 0.5 else 0 
            
            all_probabilities.append(phishing_prob) 
            all_predictions.append(prediction) 
            all_labels.append(item['labels'].item()) 
    
    
    y_true = np.array(all_labels) # convert the list of labels to a numpy array
    y_pred = np.array(all_predictions) 
    y_proba = np.array(all_probabilities)
    
   
    accuracy = accuracy_score(y_true, y_pred) 
    precision = precision_score(y_true, y_pred) 
    recall = recall_score(y_true, y_pred) 
    f1 = f1_score(y_true, y_pred) 
    auc = roc_auc_score(y_true, y_proba) 
    
    print(f"\nAnalysis:")
    print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"   Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"   Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"   F1-Score:  {f1:.4f}")
    print(f"   AUC:       {auc:.4f}")
    print("\n")
    
    cm = confusion_matrix(y_true, y_pred) # calculate the confusion matrix

    print(f"   True Positives:  {cm[1,1]:4d}")
    print(f"   True Negatives:  {cm[0,0]:4d}")
    print(f"   False Positives: {cm[0,1]:4d}")
    print(f"   False Negatives: {cm[1,0]:4d}")
    print(f"   False Positive Rate: {cm[0,1]/(cm[0,0]+cm[0,1]):.4f}")
    print(f"   False Negative Rate: {cm[1,0]/(cm[1,1]+cm[1,0]):.4f}")
    
    return y_true, y_pred, y_proba, {
        'accuracy': accuracy,
        'precision': precision, 
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'confusion_matrix': cm
    }

if 'model' in locals() and 'test_dataset' in locals(): # check if the model and test dataset are available
    y_true, y_pred, y_proba, metrics = evaluate_bert_model(model, tokenizer, test_dataset, test_df) 
    print(f"\nEvaluation completed!")
else:
    print("Warning: Model or test dataset not available. Please run training cells first.")


Model's evaluation

Analysis:
   Accuracy:  0.9753 (97.53%)
   Precision: 0.9586 (95.86%)
   Recall:    0.9795 (97.95%)
   F1-Score:  0.9690
   AUC:       0.9974


   True Positives:  1436
   True Negatives:  2202
   False Positives:   62
   False Negatives:   30
   False Positive Rate: 0.0274
   False Negative Rate: 0.0205

Evaluation completed!


## 8. Threshold optimisation and three-class metrics


In [12]:
def find_optimal_thresholds(y_true, y_proba, target_recall=0.98, target_precision=0.985):
 
  
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)   # Calculate precision-recall curve
    
    # Find threshold for target recall (lower threshold)
    recall_mask = recall >= target_recall # create a mask for the recall values that are greater than or equal to the target recall
    if np.any(recall_mask): 
        valid_indices = np.where(recall_mask)[0] 
        
        best_recall_idx = valid_indices[np.argmax(precision[valid_indices])] 
        lower_threshold = thresholds[best_recall_idx] if best_recall_idx < len(thresholds) else 0.0 
        achieved_recall = recall[best_recall_idx]  # Recall at the best threshold
        recall_precision = precision[best_recall_idx] 
    else:
        lower_threshold = 0.0 
        achieved_recall = recall[0] if len(recall) > 0 else 0.0 
        recall_precision = precision[0] if len(precision) > 0 else 0.0 
    
    # Find threshold for target precision (upper threshold)
    precision_mask = precision >= target_precision 
    if np.any(precision_mask): 
        valid_indices = np.where(precision_mask)[0] 
        
        # Choose threshold with highest recall among valid precision options
        best_precision_idx = valid_indices[np.argmax(recall[valid_indices])]
        upper_threshold = thresholds[best_precision_idx] if best_precision_idx < len(thresholds) else 1.0
        achieved_precision = precision[best_precision_idx] # Precision at the best threshold
        precision_recall = recall[best_precision_idx]
    else:
        upper_threshold = 1.0
        achieved_precision = precision[-1] if len(precision) > 0 else 0.0
        precision_recall = recall[-1] if len(recall) > 0 else 0.0
    
    print(f"   Target Recall: {target_recall*100:.1f}%")
    print(f"   Target Precision: {target_precision*100:.1f}%")
    
    print(f"   Lower threshold: {lower_threshold:.4f} (for {target_recall*100:.1f}% recall)")
    print(f"   Upper threshold: {upper_threshold:.4f} (for {target_precision*100:.1f}% precision)")
    
    print(f"\nPerformance:")
    print(f"   At lower threshold: Recall={achieved_recall:.4f} ({achieved_recall*100:.2f}%), Precision={recall_precision:.4f} ({recall_precision*100:.2f}%)")
    print(f"   At upper threshold: Precision={achieved_precision:.4f} ({achieved_precision*100:.2f}%), Recall={precision_recall:.4f} ({precision_recall*100:.2f}%)")
    
    return lower_threshold, upper_threshold

    # Define a function to calculate metrics for three-class classification system
def calculate_three_class_metrics(y_true, y_proba, lower_threshold, upper_threshold):
    
    # Apply three-class classification
    predictions = np.full(len(y_proba), 'human_review') # Initialize all predictions as human review
    predictions[y_proba <= lower_threshold] = 'safe' # Set safe if probability is less than or equal to lower threshold
    predictions[y_proba >= upper_threshold] = 'phishing' # Set phishing if probability is greater than or equal to upper threshold
    
    # Calculate the number of safe, phishing, and human review emails
    safe_count = np.sum(predictions == 'safe')
    phishing_count = np.sum(predictions == 'phishing')
    human_review_count = np.sum(predictions == 'human_review')
    total = len(predictions)
    
    # Calculate error rates
    safe_mask = (y_true == 0)
    phishing_mask = (y_true == 1)
    
    # FPR: safe emails classified as phishing
    fpr = np.sum((predictions == 'phishing') & safe_mask) / np.sum(safe_mask) if np.sum(safe_mask) > 0 else 0
    
    # FNR: phishing emails classified as safe
    fnr = np.sum((predictions == 'safe') & phishing_mask) / np.sum(phishing_mask) if np.sum(phishing_mask) > 0 else 0
    
    # Human review rate
    hr_rate = human_review_count / total
    
    # Automation rate
    automation_rate = (safe_count + phishing_count) / total
    
    print(f"Analysis")
    print(f"   Safe emails:     {safe_count:5d} ({safe_count/total*100:5.1f}%)")
    print(f"   Phishing emails: {phishing_count:5d} ({phishing_count/total*100:5.1f}%)")
    print(f"   Human review:    {human_review_count:5d} ({hr_rate*100:5.1f}%)")
    print(f"   Total:           {total:5d}")
    
    print(f"\n")
    print(f"   False Positive Rate: {fpr:.4f} ({fpr*100:.2f}%)")
    print(f"   False Negative Rate: {fnr:.4f} ({fnr*100:.2f}%)")
    print(f"   Human Review Rate:   {hr_rate:.4f} ({hr_rate*100:.2f}%)")
    print(f"   Automation Rate:     {automation_rate:.4f} ({automation_rate*100:.2f}%)")
    
    return {
        'fpr': fpr,
        'fnr': fnr,
        'human_review_rate': hr_rate,
        'automation_rate': automation_rate,
        'safe_count': safe_count,
        'phishing_count': phishing_count,
        'human_review_count': human_review_count
    }

# check if the y_true and y_proba are available
if 'y_true' in locals() and 'y_proba' in locals():
    # Find optimal thresholds
    lower_thresh, upper_thresh = find_optimal_thresholds(y_true, y_proba, target_recall=0.98, target_precision=0.985)
    
    # Calculate three-class metrics
    three_class_metrics = calculate_three_class_metrics(y_true, y_proba, lower_thresh, upper_thresh)
    

    print(f"Lower threshold: {lower_thresh:.4f}")
    print(f"Upper threshold: {upper_thresh:.4f}")
    print(f"Human review rate: {three_class_metrics['human_review_rate']*100:.1f}%")
    
else:
    print("Warning: Please run the evaluation cell first to get predictions.")
   


   Target Recall: 98.0%
   Target Precision: 98.5%
   Lower threshold: 0.4756 (for 98.0% recall)
   Upper threshold: 0.8187 (for 98.5% precision)

Performance:
   At lower threshold: Recall=0.9802 (98.02%), Precision=0.9586 (95.86%)
   At upper threshold: Precision=0.9855 (98.55%), Recall=0.9277 (92.77%)
Analysis
   Safe emails:      2232 ( 59.8%)
   Phishing emails:  1380 ( 37.0%)
   Human review:      118 (  3.2%)
   Total:            3730


   False Positive Rate: 0.0088 (0.88%)
   False Negative Rate: 0.0205 (2.05%)
   Human Review Rate:   0.0316 (3.16%)
   Automation Rate:     0.9684 (96.84%)
Lower threshold: 0.4756
Upper threshold: 0.8187
Human review rate: 3.2%


## 9. Inference


In [18]:
LOWER_THRESHOLD = 0.4756  
UPPER_THRESHOLD = 0.8187

def classify_email_bert(email_text, model=model, tokenizer=tokenizer):
    
     
    
    try:
        # Tokenize the email
        inputs = tokenizer(
            email_text,
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )
        
        # Get prediction probability
        model.eval()
        with torch.no_grad():
            outputs = model(**inputs)
            probabilities = torch.softmax(outputs.logits, dim=1) 
            phishing_prob = probabilities[0][1].item()
        
        # Apply three-threshold classification
        if phishing_prob <= LOWER_THRESHOLD:
            decision = "safe"
            confidence = "high" if phishing_prob <= 0.01 else "medium"
        elif phishing_prob >= UPPER_THRESHOLD:
            decision = "phishing" 
            confidence = "high" if phishing_prob >= 0.99 else "medium"
        else:
            decision = "human_review"
            confidence = "uncertain"
        
        return {
            "email_text": email_text[:100] + "..." if len(email_text) > 100 else email_text,
            "decision": decision,
            "probability": round(phishing_prob, 4),
            "confidence": confidence,
            "thresholds": {"lower": LOWER_THRESHOLD, "upper": UPPER_THRESHOLD}
        }
        
    except Exception as e:
        return {"error": f"Classification failed: {str(e)}"}

# Test emails
sample_emails = [
    "Thank you for your interest in our services. Please visit our website for more information.",
    "URGENT: Your account has been compromised! Click here immediately: http://fake-bank-login.com/urgent",
    "Hi there, I hope this email finds you well. I wanted to follow up on our previous conversation.",
    "CONGRATULATIONS! You've won $1,000,000! Send us your bank details to claim your prize!"
]

print("Results:")

for i, email in enumerate(sample_emails, 1):
    result = classify_email_bert(email)
    if "error" not in result:
        print(f"Email {i}: {result['email_text']}")
        print(f"Decision: {result['decision']} ({result['confidence']} confidence)")
        print(f"Probability: {result['probability']}")
        print("-" * 50)
    else:
        print(f"Error: {result['error']}")
        break

print(f"Thresholds: Lower={LOWER_THRESHOLD}, Upper={UPPER_THRESHOLD}")
print("Classifications: safe | human_review | phishing")

# set as default classifier
classify_email = classify_email_bert

Results:
Email 1: Thank you for your interest in our services. Please visit our website for more information.
Decision: phishing (medium confidence)
Probability: 0.9883
--------------------------------------------------
Email 2: URGENT: Your account has been compromised! Click here immediately: http://fake-bank-login.com/urgent
Decision: phishing (high confidence)
Probability: 0.9989
--------------------------------------------------
Email 3: Hi there, I hope this email finds you well. I wanted to follow up on our previous conversation.
Decision: safe (medium confidence)
Probability: 0.0145
--------------------------------------------------
Email 4: CONGRATULATIONS! You've won $1,000,000! Send us your bank details to claim your prize!
Decision: phishing (high confidence)
Probability: 0.9947
--------------------------------------------------
Thresholds: Lower=0.4756, Upper=0.8187
Classifications: safe | human_review | phishing
